In [2]:
import os
from moviepy import VideoFileClip, AudioFileClip  # <-- Updated import for v2.0
from scipy.io import wavfile
import noisereduce as nr

/Users/apple/Code/utmtvenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def clean_video_audio(input_video_path, output_video_path, noise_reduction_amount=0.8):
    temp_raw_audio = "temp_raw_audio.wav"
    temp_clean_audio = "temp_clean_audio.wav"
    
    print("Step 1: Extracting audio from video...")
    video = VideoFileClip(input_video_path)
    
    # Extract original audio as WAV
    video.audio.write_audiofile(temp_raw_audio, codec='pcm_s16le')
    
    print("Step 2: Processing audio to remove background noise...")
    rate, data = wavfile.read(temp_raw_audio)
    
    # Handle stereo audio shapes for noisereduce
    is_stereo = len(data.shape) > 1
    if is_stereo:
        data = data.T
        
    reduced_noise = nr.reduce_noise(y=data, 
                                    sr=rate, 
                                    prop_decrease=noise_reduction_amount, 
                                    stationary=False)
    
    if is_stereo:
        reduced_noise = reduced_noise.T
        
    wavfile.write(temp_clean_audio, rate, reduced_noise)
    
    print("Step 3: Stitching cleaned audio back into video...")
    cleaned_audio_clip = AudioFileClip(temp_clean_audio)
    
    # <-- Updated method: set_audio() is now with_audio() in v2.0
    final_video = video.with_audio(cleaned_audio_clip)
    
    final_video.write_videofile(output_video_path, 
                                codec="libx264", 
                                audio_codec="aac")
    
    print("Step 4: Cleaning up temporary files...")
    video.close()
    cleaned_audio_clip.close()
    
    if os.path.exists(temp_raw_audio): os.remove(temp_raw_audio)
    if os.path.exists(temp_clean_audio): os.remove(temp_clean_audio)
        
    print(f"Success! Cleaned video saved to: {output_video_path}")

In [ ]:
# pip install torch==2.0.1 torchaudio==2.0.2 deepfilternet

import os
import shutil
import subprocess
from moviepy import VideoFileClip, AudioFileClip

def ai_clean_video(input_video_path, output_video_path):
    """
    Extracts audio, processes it through a Deep Neural Network (DeepFilterNet) 
    for studio-quality speech extraction, and stitches it back.
    """
    temp_raw_audio = "temp_raw_audio.wav"
    out_dir = "ai_audio_out"
    
    print("Step 1: Extracting audio...")
    video = VideoFileClip(input_video_path)
    video.audio.write_audiofile(temp_raw_audio, codec='pcm_s16le')
    
    print("Step 2: Applying AI Speech Enhancement (DeepFilterNet)...")
    # This triggers the DeepFilterNet AI to process the WAV file.
    # It automatically creates the 'out_dir' folder and saves the result there.
    try:
        subprocess.run(["deepFilter", temp_raw_audio, "-o", out_dir], check=True)
    except FileNotFoundError:
        print("Error: DeepFilter command not found. Did you run 'pip install deepfilternet'?")
        return
        
    # DeepFilterNet automatically appends '_DeepFilterNet3' to the output filename
    clean_audio_file = os.path.join(out_dir, "temp_raw_audio_DeepFilterNet3.wav")
    
    print("Step 3: Stitching AI-cleaned audio back into video...")
    clean_audio = AudioFileClip(clean_audio_file)
    final_video = video.with_audio(clean_audio)
    
    # Write final video
    final_video.write_videofile(output_video_path, 
                                codec="libx264", 
                                audio_codec="aac")
    
    print("Step 4: Cleaning up temporary files...")
    video.close()
    clean_audio.close()
    
    # Delete the raw extraction and the AI output folder
    if os.path.exists(temp_raw_audio): os.remove(temp_raw_audio)
    if os.path.exists(out_dir): shutil.rmtree(out_dir)
        
    print(f"Success! Studio-quality video saved to: {output_video_path}")

In [ ]:
input_file = "sample_videos/noisy_video2.mp4"    
output_file = "sample_videos/clean_video2.mp4"   

# clean_video_audio(input_file, output_file, noise_reduction_amount=0.7)
ai_clean_video(input_file, output_file)

Step 1: Extracting audio...
MoviePy - Writing audio in temp_raw_audio.wav


MoviePy - Done.
Step 2: Applying AI Speech Enhancement (DeepFilterNet)...
2026-06-30 14:26:45 | INFO     | DF | Running on torch 2.0.1
2026-06-30 14:26:45 | INFO     | DF | Running on host MacBookAir.lan
2026-06-30 14:26:45 | INFO     | DF | Loading model settings of DeepFilterNet3
2026-06-30 14:26:45 | INFO     | DF | Using DeepFilterNet3 model at /Users/apple/Library/Caches/DeepFilterNet/DeepFilterNet3
2026-06-30 14:26:45 | INFO     | DF | Initializing model `deepfilternet3`
2026-06-30 14:26:45 | INFO     | DF | Found checkpoint /Users/apple/Library/Caches/DeepFilterNet/DeepFilterNet3/checkpoints/model_120.ckpt.best with epoch 120
2026-06-30 14:26:45 | INFO     | DF | Running on device cpu
2026-06-30 14:26:45 | INFO     | DF | Model loaded


fatal: not a git repository (or any of the parent directories): .git
2026-06-30 14:26:46.551 | WARNING  | df.logger:warn_once:75 - Audio sampling rate does not match model sampling rate (44100, 48000). Resampling...
/Users/apple/Code/utmtvenv/lib/python3.11/site-packages/torchaudio/functional/functional.py:1458: UserWarning: "sinc_interpolation" resampling method name is being deprecated and replaced by "sinc_interp_hann" in the next release. The default behavior remains unchanged.
  warnings.warn(


2026-06-30 14:26:52 | INFO     | DF | Enhanced noisy audio file 'temp_raw_audio.wav' in 5.62s (RT factor: 0.080)


/Users/apple/Code/utmtvenv/lib/python3.11/site-packages/torchaudio/functional/functional.py:1458: UserWarning: "sinc_interpolation" resampling method name is being deprecated and replaced by "sinc_interp_hann" in the next release. The default behavior remains unchanged.
  warnings.warn(


Step 3: Stitching AI-cleaned audio back into video...
MoviePy - Building video sample_videos/clean_video1.mp4.
MoviePy - Writing audio in clean_video1TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video sample_videos/clean_video1.mp4



MoviePy - Done !
MoviePy - video ready sample_videos/clean_video1.mp4
Step 4: Cleaning up temporary files...
Success! Studio-quality video saved to: sample_videos/clean_video1.mp4
